In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import kendalltau
from itertools import combinations
import re
import matplotlib.pyplot as plt

import mne
from mne.viz import circular_layout
from mne_connectivity.viz import plot_connectivity_circle
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Calibri'] + plt.rcParams['font.sans-serif']

csv = pd.read_csv('/path/to/avg_ig_by_region.csv')
lookup = pd.read_csv('/path/to/fs_lookup.csv', usecols=['code', 'region'])
lobes = pd.read_csv('/path/to/destrieux - lobe.csv')

VALUE_COL = 'avg_saliency'

lobe = list(lobes['lobe'])
area = list(lobes['area'])
lharea = ['lh_' + i for i in area]
rharea = ['rh_' + i for i in area]
rharea = rharea[::-1]
area = lharea + rharea
lobe = lobe + lobe[::-1]

labels_ref = area
colors = lobe

data = csv.pivot_table(index='subject', columns='region_id',
                       values=VALUE_COL, aggfunc='first', dropna=False)
data.columns = data.columns.astype(int)

code_to_region = dict(zip(lookup['code'], lookup['region']))
data.columns = [re.sub(r'^ctx_', '', code_to_region.get(c, str(c))) for c in data.columns]
data = data.apply(pd.to_numeric, errors='coerce')

EXCLUDE = {'medial_wall', 'unknown'}   

def is_excluded(region_name):
    return re.sub(r'^(lh|rh)_', '', region_name).lower() in EXCLUDE

available = [l for l in labels_ref if l in data.columns and not is_excluded(l)]
missing   = [l for l in labels_ref if l not in data.columns]

dropped = [l for l in labels_ref if l in data.columns and is_excluded(l)]
if dropped:
    print(f"Excluded {len(dropped)} regions:", dropped)
if missing:
    print(f"{len(missing)} regions in labels_ref not found in data — dropped:", missing)

data = data[available]     
labels = available         
n = len(labels)

connectivity_array = np.zeros((n, n))

for i, j in combinations(range(n), 2):
    xi, yj = labels[i], labels[j]
    pair = data[[xi, yj]].dropna()
    if len(pair) < 3:
        continue
    tau, pvalue = kendalltau(pair[xi], pair[yj])
    if pvalue < 0.05 and tau > 0:
        connectivity_array[i, j] = abs(tau)
        connectivity_array[j, i] = abs(tau)

area_to_lobe = dict(zip(lobes['area'], lobes['lobe']))

def strip_hemisphere(region_name):
    return re.sub(r'^(lh|rh)_', '', region_name)   

def get_rgba_value(number):
    if str(number) == "frontal":
        return (234/255,67/255,53/255, 255/255)
    elif str(number) == "parietal":
        return (227/255,116/255,0/255, 255/255)
    elif str(number) == "occipital":
        return (66/255,103/255,210/255, 255/255)
    elif str(number) == "temporal":
        return (52/255,168/255,83/255, 255/255)
    elif str(number) == "limbic":
        return (255/255,194/255,0/255, 255/255)
    elif int(number) == 5:
        return (251/255,188/255,4/255,255/255)

colors = [get_rgba_value(c) for c in colors]

new_lobe_order = ['frontal', 'temporal', 'parietal', 'occipital', 'limbic']

def region_lobe(region_name):
    return area_to_lobe.get(strip_hemisphere(region_name))

lh_labels = [l for l in labels if l.startswith('lh_')]
rh_labels = [l for l in labels if l.startswith('rh_')]

lh_sorted = []
for lobe in new_lobe_order:
    lh_sorted += [l for l in lh_labels if region_lobe(l) == lobe]

rh_sorted = []
for lobe in reversed(new_lobe_order):
    rh_sorted += [l for l in rh_labels if region_lobe(l) == lobe]

labels = lh_sorted + rh_sorted
colors = [get_rgba_value(region_lobe(l)) for l in labels]

data = data[labels]
n = len(labels)
connectivity_array = np.zeros((n, n))

for i, j in combinations(range(n), 2):
    xi, yj = labels[i], labels[j]
    pair = data[[xi, yj]].dropna()
    if len(pair) < 3:
        continue
    tau, pvalue = kendalltau(pair[xi], pair[yj])
    if pvalue < 0.05 and tau > 0:
        connectivity_array[i, j] = abs(tau)
        connectivity_array[j, i] = abs(tau)

In [ ]:
from statsmodels.stats.multitest import multipletests

def build_connectivity(data, labels, alpha=0.05, method='fdr_bh',
                       positive_only=True, min_n=3):
    n = len(labels)
    pairs, taus, pvals = [], [], []

    for i, j in combinations(range(n), 2):
        xi, yj = labels[i], labels[j]
        pair = data[[xi, yj]].dropna()
        if len(pair) < min_n:
            continue
        tau, p = kendalltau(pair[xi], pair[yj])
        if np.isnan(p):
            continue
        pairs.append((i, j))
        taus.append(tau)
        pvals.append(p)

    taus = np.asarray(taus)
    pvals = np.asarray(pvals)

    reject, qvals, _, _ = multipletests(pvals, alpha=alpha, method=method)

    keep = reject & (taus > 0) if positive_only else reject

    conn = np.zeros((n, n))
    for (i, j), t, k in zip(pairs, taus, keep):
        if k:
            conn[i, j] = conn[j, i] = abs(t)

    print(f"{keep.sum()} / {len(pvals)} pairs survive {method} at q<{alpha}")
    return conn, pairs, taus, pvals, qvals

data = data[labels]
n = len(labels)
connectivity_array, pairs, taus, pvals, qvals = build_connectivity(data, labels)

In [ ]:
import numpy as np

TAUS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
TOP_N = 50

conn = np.asarray(connectivity_array, dtype=float)

for t in TAUS:
    adj = connectivity_array > t
    np.fill_diagonal(adj, False)
    deg = adj.sum(axis=1)
    order = np.argsort(-deg, kind="stable")[:TOP_N]
    print(f"\n=== tau > {t:.1f} ===")
    for rank, i in enumerate(order, 1):
        print(f"{rank:>3}. {str(labels[i]):<40} {deg[i]}")

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

TAUS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
TOP_N = 50
PIE_TAU = 0.5
ERROR_BARS = "sem"          

NETWORKS = {
    "lh_G_Ins_lg_and_S_cent_ins": ["Salience"],
    "lh_G_and_S_cingul-Ant": ["DMN", "Limbic", "Salience", "Social"],
    "lh_G_and_S_cingul-Mid-Ant": ["DMN", "FPN", "Salience", "Sensorimotor"],
    "lh_G_and_S_cingul-Mid-Post": ["DMN"],
    "lh_G_and_S_frontomargin": ["DMN", "Language", "Salience"],
    "lh_G_and_S_occipital_inf": ["Dorsal/Ventral Attention"],
    "lh_G_and_S_subcentral": ["Sensorimotor"],
    "lh_G_and_S_transv_frontopol": ["DMN"],
    "lh_G_cingul-Post-dorsal": ["DMN", "FPN"],
    "lh_G_cuneus": ["DMN", "Dorsal/Ventral Attention", "FPN", "Language", "Salience"],
    "lh_G_front_inf-Opercular": ["Language"],
    "lh_G_front_inf-Orbital": ["Language", "Limbic"],
    "lh_G_front_inf-Triangul": ["FPN", "Language"],
    "lh_G_front_middle": ["Dorsal/Ventral Attention"],
    "lh_G_front_sup": ["DMN", "FPN", "Salience"],
    "lh_G_insular_short": ["Salience"],
    "lh_G_oc-temp_lat-fusifor": ["Dorsal/Ventral Attention"],
    "lh_G_oc-temp_med-Lingual": ["Dorsal/Ventral Attention"],
    "lh_G_oc-temp_med-Parahip": ["DMN", "Limbic"],
    "lh_G_occipital_middle": ["Dorsal/Ventral Attention"],
    "lh_G_occipital_sup": ["DMN", "Dorsal/Ventral Attention", "Salience"],
    "lh_G_orbital": ["FPN", "Limbic"],
    "lh_G_pariet_inf-Angular": ["DMN", "Language"],
    "lh_G_pariet_inf-Supramar": ["DMN", "Dorsal/Ventral Attention", "FPN", "Language"],
    "lh_G_parietal_sup": ["Dorsal/Ventral Attention", "FPN", "Sensorimotor"],
    "lh_G_postcentral": ["Sensorimotor"],
    "lh_G_precentral": ["Sensorimotor"],
    "lh_G_precuneus": ["DMN", "Limbic", "Social"],
    "lh_G_rectus": ["DMN", "Limbic", "Social"],
    "lh_G_subcallosal": ["DMN", "Limbic", "Salience"],
    "lh_G_temp_sup-G_T_transv": ["Language", "Salience"],
    "lh_G_temp_sup-Lateral": ["DMN", "Dorsal/Ventral Attention", "Language"],
    "lh_G_temp_sup-Plan_polar": ["DMN", "Language", "Limbic"],
    "lh_G_temp_sup-Plan_tempo": ["Language"],
    "lh_G_temporal_inf": ["DMN", "Language", "Social"],
    "lh_G_temporal_middle": ["DMN", "FPN", "Language", "Social"],
    "lh_Lat_Fis-ant-Horizont": ["FPN", "Language"],
    "lh_Lat_Fis-ant-Vertical": ["Language", "Salience"],
    "lh_Lat_Fis-post": ["DMN", "Dorsal/Ventral Attention", "Language"],
    "lh_Pole_occipital": ["Dorsal/Ventral Attention"],
    "lh_S_calcarine": ["Dorsal/Ventral Attention"],
    "lh_S_central": ["Sensorimotor"],
    "lh_S_cingul-Marginalis": ["DMN", "Sensorimotor"],
    "lh_S_circular_insula_ant": ["Salience"],
    "lh_S_circular_insula_inf": ["Limbic", "Salience", "Sensorimotor"],
    "lh_S_circular_insula_sup": ["Salience"],
    "lh_S_collat_transv_ant": ["DMN", "Limbic"],
    "lh_S_front_inf": ["FPN", "Language"],
    "lh_S_front_middle": ["FPN"],
    "lh_S_front_sup": ["DMN", "FPN"],
    "lh_S_interm_prim-Jensen": ["DMN", "FPN"],
    "lh_S_intrapariet_and_P_trans": ["Dorsal/Ventral Attention", "FPN", "Salience"],
    "lh_S_oc_middle_and_Lunatus": ["Dorsal/Ventral Attention"],
    "lh_S_oc_sup_and_transversal": ["Dorsal/Ventral Attention"],
    "lh_S_oc-temp_lat": ["Dorsal/Ventral Attention"],
    "lh_S_oc-temp_med_and_Lingual": ["Dorsal/Ventral Attention"],
    "lh_S_occipital_ant": ["Dorsal/Ventral Attention"],
    "lh_S_orbital-H_Shaped": ["FPN", "Limbic"],
    "lh_S_orbital_lateral": ["Dorsal/Ventral Attention", "Limbic", "Salience"],
    "lh_S_orbital_med-olfact": ["DMN", "Limbic"],
    "lh_S_parieto_occipital": ["DMN", "Dorsal/Ventral Attention"],
    "lh_S_pericallosal": ["DMN", "Limbic"],
    "lh_S_postcentral": ["Sensorimotor"],
    "lh_S_precentral-inf-part": ["Language", "Sensorimotor"],
    "lh_S_precentral-sup-part": ["Dorsal/Ventral Attention", "Sensorimotor"],
    "lh_S_suborbital": ["Limbic"],
    "lh_S_subparietal": ["DMN"],
    "lh_S_temporal_inf": ["DMN", "Language"],
    "lh_S_temporal_sup": ["Language", "Social"],
    "lh_S_temporal_transverse": ["Language"],
    "rh_G_Ins_lg_and_S_cent_ins": ["Salience", "Sensorimotor"],
    "rh_G_and_S_cingul-Ant": ["DMN", "FPN", "Limbic", "Salience"],
    "rh_G_and_S_cingul-Mid-Ant": ["FPN", "Limbic", "Salience"],
    "rh_G_and_S_cingul-Mid-Post": ["FPN", "Limbic", "Salience", "Sensorimotor"],
    "rh_G_and_S_frontomargin": ["FPN"],
    "rh_G_and_S_occipital_inf": ["Dorsal/Ventral Attention"],
    "rh_G_and_S_subcentral": ["Sensorimotor"],
    "rh_G_and_S_transv_frontopol": ["DMN"],
    "rh_G_cingul-Post-dorsal": ["DMN", "FPN"],
    "rh_G_front_inf-Opercular": ["Dorsal/Ventral Attention", "FPN", "Salience"],
    "rh_G_front_inf-Orbital": ["Language", "Limbic", "Social"],
    "rh_G_front_middle": ["Dorsal/Ventral Attention", "FPN"],
    "rh_G_front_sup": ["DMN", "FPN"],
    "rh_G_insular_short": ["Salience"],
    "rh_G_oc-temp_med-Parahip": ["DMN", "Limbic"],
    "rh_G_orbital": ["DMN", "Limbic", "Salience"],
    "rh_G_pariet_inf-Angular": ["DMN"],
    "rh_G_pariet_inf-Supramar": ["Dorsal/Ventral Attention", "Limbic", "Social"],
    "rh_G_parietal_sup": ["DMN"],
    "rh_G_postcentral": ["Sensorimotor"],
    "rh_G_precentral": ["Sensorimotor"],
    "rh_G_precuneus": ["DMN", "FPN"],
    "rh_G_rectus": ["DMN", "Limbic"],
    "rh_G_subcallosal": ["DMN", "Limbic", "Salience"],
    "rh_G_temporal_middle": ["DMN", "Language", "Social"],
    "rh_G_temp_sup-G_T_transv": ["Sensorimotor", "Language"],
    "rh_G_temp_sup-Lateral": ["Language"],
    "rh_G_temp_sup-Plan_polar": ["DMN", "Language", "Limbic"],
    "rh_G_temp_sup-Plan_tempo": ["Dorsal/Ventral Attention", "Language"],
    "rh_Lat_Fis-ant-Horizont": ["Dorsal/Ventral Attention", "Language", "Salience"],
    "rh_Lat_Fis-ant-Vertical": ["Dorsal/Ventral Attention", "FPN"],
    "rh_Lat_Fis-post": ["DMN", "Dorsal/Ventral Attention", "Sensorimotor"],
    "rh_Pole_occipital": ["Dorsal/Ventral Attention"],
    "rh_Pole_temporal": ["DMN", "Social", "Limbic", "Language"],
    "rh_S_central": ["Sensorimotor"],
    "rh_S_cingul-Marginalis": ["DMN", "Salience", "Sensorimotor"],
    "rh_S_circular_insula_ant": ["Dorsal/Ventral Attention", "Salience"],
    "rh_S_circular_insula_inf": ["Limbic", "Salience", "Sensorimotor"],
    "rh_S_circular_insula_sup": ["Dorsal/Ventral Attention", "Salience"],
    "rh_S_collat_transv_ant": ["DMN", "Language"],
    "rh_S_front_inf": ["Dorsal/Ventral Attention", "FPN", "Language"],
    "rh_S_front_middle": ["Dorsal/Ventral Attention", "FPN"],
    "rh_S_front_sup": ["DMN", "Dorsal/Ventral Attention", "FPN"],
    "rh_S_interm_prim-Jensen": ["DMN", "Dorsal/Ventral Attention", "FPN"],
    "rh_S_intrapariet_and_P_trans": ["Dorsal/Ventral Attention", "FPN"],
    "rh_S_oc-temp_med_and_Lingual": ["Dorsal/Ventral Attention"],
    "rh_S_oc_middle_and_Lunatus": ["Dorsal/Ventral Attention"],
    "rh_S_orbital-H_Shaped": ["DMN", "Limbic", "Salience"],
    "rh_S_orbital_med-olfact": ["Limbic"],
    "rh_S_parieto_occipital": ["DMN", "Dorsal/Ventral Attention"],
    "rh_S_pericallosal": ["DMN", "Limbic", "Salience"],
    "rh_S_postcentral": ["FPN", "Sensorimotor"],
    "rh_S_precentral-inf-part": ["Dorsal/Ventral Attention", "FPN", "Language", "Sensorimotor"],
    "rh_S_precentral-sup-part": ["Sensorimotor"],
    "rh_S_suborbital": ["DMN", "Limbic", "Salience"],
    "rh_S_subparietal": ["DMN", "FPN"],
    "rh_S_temporal_sup": ["Language", "Social"],
}

sns.set_theme(style='whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Calibri'] + plt.rcParams['font.sans-serif']
plt.rcParams['axes.edgecolor'] = '#444444'
plt.rcParams['axes.labelcolor'] = '#222222'
plt.rcParams['text.color'] = '#222222'
plt.rcParams['xtick.color'] = '#333333'
plt.rcParams['ytick.color'] = '#333333'

TAUS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

ORDER = ['DMN', 'Dorsal/Ventral Attention', 'FPN', 'Salience',
         'Sensorimotor', 'Language', 'Limbic', 'Social']
LABELS = {'DMN': 'Default Mode', 'FPN': 'Frontoparietal', 'Limbic': 'Limbic/Emotion',
          'Dorsal/Ventral Attention': 'Dorsal/Ventral'}
COLORS = {'DMN': '#4C72B0', 'Dorsal/Ventral Attention': '#DD8452', 'FPN': '#55A868',
          'Salience': '#C44E52', 'Sensorimotor': '#8172B3', 'Language': '#937860',
          'Limbic': '#DA8BC3', 'Social': '#8C8C8C'}
HEMI = {'lh': '#4C72B0', 'rh': '#C44E52'}

In [ ]:
conn = np.asarray(conn, dtype=float)
assert conn.shape[0] == len(labels), (conn.shape, len(labels))

rows = []
for t in TAUS:
    adj = conn > t
    np.fill_diagonal(adj, False)
    deg = adj.sum(axis=1)
    for rank, i in enumerate(np.argsort(-deg, kind="stable")[:TOP_N], 1):
        rows.append({"tau": t, "rank": rank, "region": str(labels[i]), "count": int(deg[i])})

top = pd.DataFrame(rows)
top["hemisphere"] = top.region.str[:2].str.upper()

missing = sorted(set(top.region) - set(NETWORKS))
if missing:
    raise SystemExit("no network assigned: " + ", ".join(missing))

print(top.groupby("tau")["count"].agg(["max", "min"]))
print("unique regions across all thresholds:", top.region.nunique())

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

sns.set_theme(style='whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.edgecolor'] = '#444444'
plt.rcParams['axes.labelcolor'] = '#222222'
plt.rcParams['text.color'] = '#222222'
plt.rcParams['xtick.color'] = '#333333'
plt.rcParams['ytick.color'] = '#333333'

PIE_TAU = 0.5

COLORS = {'DMN': '#4C72B0', 'Dorsal/Ventral Attention': '#DD8452', 'FPN': '#55A868',
          'Salience': '#C44E52', 'Sensorimotor': '#8172B3', 'Language': '#937860',
          'Limbic': '#DA8BC3', 'Social': '#8C8C8C'}
LABELS = {'DMN': 'Default Mode', 'FPN': 'Frontoparietal', 'Limbic': 'Limbic/Emotion',
          'Dorsal/Ventral Attention': 'Dorsal/Ventral'}

def tally(regions):
    c = {}
    for r in regions:
        for n in NETWORKS[r]:
            c[n] = c.get(n, 0) + 1
    return c

cA = pd.Series(tally(top[top.tau == PIE_TAU].region)).sort_values(ascending=False)
pA = cA / cA.sum() * 100

n = len(cA)
xp = np.arange(n) + 0.5

fig, ax = plt.subplots(figsize=(13, 13))
for i in range(n):
    if i % 2 == 0:
        ax.axvspan(i, i + 1, color='gray', alpha=0.05, zorder=0)

ax.bar(xp, cA.values, width=0.72,
       color=[COLORS.get(x, '#BBBBBB') for x in cA.index],
       edgecolor='#444444', linewidth=1.2, zorder=3)

for x, c, p in zip(xp, cA.values, pA.values):
    ax.text(x, c + cA.max() * 0.015, f'{c}\n({p:.1f}%)',
            ha='center', va='bottom', fontsize=20, color='#333333')

ax.set_xlim(0, n)
ax.set_ylim(0, cA.max() * 1.22)
ax.set_xticks(xp)
ax.set_xticklabels([LABELS.get(x, x) for x in cA.index], rotation=90, ha='right', fontsize=24)
ax.set_xlabel('Functional network', fontsize=28, fontweight='bold', labelpad=12)
ax.set_ylabel('Region count', fontsize=28, fontweight='bold')
ax.tick_params(axis='y', labelsize=24)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

sns.set_theme(style='whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.edgecolor'] = '#444444'
plt.rcParams['axes.labelcolor'] = '#222222'
plt.rcParams['text.color'] = '#222222'
plt.rcParams['xtick.color'] = '#333333'
plt.rcParams['ytick.color'] = '#333333'

TAUS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
ERROR_BARS = 'sem'         
nT = len(TAUS)

COLORS = {'DMN': '#4C72B0', 'Dorsal/Ventral Attention': '#DD8452', 'FPN': '#55A868',
          'Salience': '#C44E52', 'Sensorimotor': '#8172B3', 'Language': '#937860',
          'Limbic': '#DA8BC3', 'Social': '#8C8C8C'}
LABELS = {'DMN': 'Default Mode', 'FPN': 'Frontoparietal', 'Limbic': 'Limbic/Emotion',
          'Dorsal/Ventral Attention': 'Dorsal/Ventral'}

def tally(regions):
    c = {}
    for r in regions:
        for n in NETWORKS[r]:
            c[n] = c.get(n, 0) + 1
    return c

cnt = pd.DataFrame({t: tally(g.region) for t, g in top.groupby('tau')}).fillna(0).astype(int)
pct = cnt / cnt.sum(axis=0) * 100

b = pd.DataFrame({'mean': pct.mean(axis=1), 'sd': pct.std(axis=1, ddof=1),
                  'min': pct.min(axis=1), 'max': pct.max(axis=1)})
b['sem'] = b['sd'] / np.sqrt(nT)
b = b.sort_values('mean', ascending=False)

if ERROR_BARS == 'range':
    e = np.vstack([b['mean'] - b['min'], b['max'] - b['mean']])
else:
    s = b['sd'] if ERROR_BARS == 'sd' else b['sem']
    e = np.vstack([s, s])

n = len(b)
xp = np.arange(n) + 0.5

fig, ax = plt.subplots(figsize=(13, 13))
for i in range(n):
    if i % 2 == 0:
        ax.axvspan(i, i + 1, color='gray', alpha=0.05, zorder=0)

ax.bar(xp, b['mean'], width=0.72, yerr=e, capsize=8,
       color=[COLORS.get(x, '#BBBBBB') for x in b.index],
       edgecolor='#444444', linewidth=1.2,
       error_kw={'elinewidth': 2, 'capthick': 2, 'ecolor': '#333333'}, zorder=3)

ax.set_xlim(0, n)
ax.set_xticks(xp)
ax.set_xticklabels([LABELS.get(x, x) for x in b.index], rotation=90, ha='right', fontsize=24)
ax.set_xlabel('Functional network', fontsize=28, fontweight='bold', labelpad=12)
ax.set_ylabel('Mean involvement (%)', fontsize=28, fontweight='bold')
ax.tick_params(axis='y', labelsize=24)

plt.tight_layout()
plt.show()

print(b.round(2))

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
%matplotlib inline

sns.set_theme(style='whitegrid')
sns.set_theme(style='whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.edgecolor'] = '#444444'
plt.rcParams['axes.labelcolor'] = '#222222'
plt.rcParams['text.color'] = '#222222'
plt.rcParams['xtick.color'] = '#333333'
plt.rcParams['ytick.color'] = '#333333'

TAUS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
TOP_N = 50
ERROR_BARS = 'sem'
nT = len(TAUS)

LOBE_ORDER = ['frontal', 'temporal', 'parietal', 'occipital', 'limbic',
              'insular', 'subcortical', 'unknown']


def get_rgba_value(number):
    if str(number) == "frontal":
        return (234/255, 67/255, 53/255, 255/255)
    elif str(number) == "parietal":
        return (227/255, 116/255, 0/255, 255/255)
    elif str(number) == "occipital":
        return (66/255, 103/255, 210/255, 255/255)
    elif str(number) == "temporal":
        return (52/255, 168/255, 83/255, 255/255)
    elif str(number) == "limbic":
        return (255/255, 194/255, 0/255, 255/255)
    return (0.72, 0.72, 0.72, 1.0)


lobe_info = pd.read_csv('/projectnb/nickar/sidd/destrieux - lobe.csv')  
lobe_of = dict(zip(lobe_info['area'], lobe_info['lobe']))

wide = (top.pivot_table(index='region', columns='tau', values='count', aggfunc='first')
           .reindex(columns=TAUS).fillna(0.0))
reg = pd.DataFrame({'mean': wide.sum(axis=1) / nT, 'sd': wide.std(axis=1, ddof=1),
                    'min': wide.min(axis=1), 'max': wide.max(axis=1)})
reg['sem'] = reg['sd'] / np.sqrt(nT)
reg['hemi'] = reg.index.str[:2].str.lower()
reg['area'] = reg.index.str[3:]
reg['lobe'] = reg['area'].map(lobe_of).fillna('unknown')

reg = reg.sort_values('mean', ascending=False).head(TOP_N)          
reg['lobe'] = pd.Categorical(reg['lobe'], categories=LOBE_ORDER, ordered=True)
reg = reg.sort_values(['lobe', 'hemi', 'mean'], ascending=[True, True, False])    

unmapped = sorted(set(reg.loc[reg['lobe'] == 'unknown', 'area']))
if unmapped:
    print('no lobe in CSV (grey):', unmapped)

if ERROR_BARS == 'range':
    e = np.vstack([reg['mean'] - reg['min'], reg['max'] - reg['mean']])
else:
    s = reg['sd'] if ERROR_BARS == 'sd' else reg['sem']
    e = np.vstack([s, s])

n = len(reg)
xp = np.arange(n) + 0.5

x_lobes = reg['lobe'].astype(str).tolist()
boundaries, start = [], 0
for i in range(1, n + 1):
    if i == n or x_lobes[i] != x_lobes[start]:
        boundaries.append((x_lobes[start], start, i))
        start = i

fig, ax = plt.subplots(figsize=(28, 13))

for k, (lobe_name, s_idx, e_idx) in enumerate(boundaries):
    if k % 2 == 0:
        ax.axvspan(s_idx, e_idx, color='gray', alpha=0.08, zorder=0)
    ax.text((s_idx + e_idx) / 2, 1.01, lobe_name.capitalize(),
            transform=ax.get_xaxis_transform(), ha='center', va='bottom',
            fontsize=24, fontweight='bold', color='#555555')

bars = ax.bar(xp, reg['mean'], width=0.72, yerr=e, capsize=5,
              color=[get_rgba_value(l) for l in reg['lobe']],
              edgecolor='#333333', linewidth=1.0,
              error_kw={'elinewidth': 1.5, 'capthick': 1.5, 'ecolor': '#333333'}, zorder=3)

for bar, h in zip(bars, reg['hemi']):
    if h == 'rh':
        bar.set_hatch('//')

for lobe_name, s_idx, e_idx in boundaries[1:]:
    ax.axvline(s_idx, color='#555555', linewidth=1.0, zorder=2)

ax.set_xlim(0, n)
ax.set_xticks(xp)
ax.set_xticklabels(reg.index, rotation=90, fontsize=18)
ax.set_ylabel('Mean saliency correlations per region', fontsize=28, fontweight='bold')
ax.set_xlabel('Destrieux-based cortical parcellation', fontsize=28, fontweight='bold', labelpad=12)
ax.tick_params(axis='y', labelsize=24)

handles = [Patch(facecolor='white', edgecolor='#333333', label='Left hemisphere'),
           Patch(facecolor='white', edgecolor='#333333', hatch='//', label='Right hemisphere')]
ax.legend(handles=handles, prop={'size': 20}, loc='upper right',
          frameon=True, edgecolor='#CCCCCC', framealpha=0.95)

plt.tight_layout()
plt.show()